In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb

In [5]:
df1=pd.read_parquet('D:/AHU/rdm-wach-ai/paraquet_data/raw/raw_one_year.parquet',engine="pyarrow")
df1.head(2)

,result,table,controller,site,power_factor_avg,units,current_l1,current_l2,current_l3,volts_l1_n,volts_l2_n,volts_l3_n,apparent_power_total,power_l1,power_l2,power_l3,power_total
time,,,,,,,,,,,,,,,,,
2025-11-05 06:00:00+00:00,0.0,1.961538,0.0,0.0,0.79,0.0,0.664,0.55,0.710000,232.340000,230.860000,232.360000,0.446164,0.135066,0.100593,0.116841,0.352846
2025-11-05 06:15:00+00:00,0.0,2.078947,0.0,0.0,0.79,0.0,0.670,0.55,0.718571,233.685714,232.185714,233.728571,0.450998,0.136399,0.101432,0.117955,0.355975


In [6]:
# I want time as columns not as index 
df1 = df1.reset_index()  # moves the index into a regular column

Timeseries slot

In [7]:
numeric_cols = df1.select_dtypes(include='number').columns
df_numeric = df1[numeric_cols]
df_numeric

,result,table,controller,site,power_factor_avg,units,current_l1,current_l2,current_l3,volts_l1_n,volts_l2_n,volts_l3_n,apparent_power_total,power_l1,power_l2,power_l3,power_total
0,0.0,1.961538,0.0,0.0,0.79,0.0,0.6640,0.5500,0.710000,232.340000,230.860000,232.360000,0.446164,0.135066,0.100593,0.116841,0.352846
1,0.0,2.078947,0.0,0.0,0.79,0.0,0.6700,0.5500,0.718571,233.685714,232.185714,233.728571,0.450998,0.136399,0.101432,0.117955,0.355975
2,0.0,2.000000,0.0,0.0,0.79,0.0,0.6700,0.5500,0.720000,233.928571,232.442857,234.028571,0.451970,0.136216,0.101376,0.118240,0.356162
3,0.0,2.023810,0.0,0.0,0.79,0.0,0.6700,0.5500,0.720000,234.075000,232.575000,234.087500,0.452858,0.136388,0.101520,0.118302,0.355818
4,0.0,2.000000,0.0,0.0,0.79,0.0,0.6700,0.5500,0.720000,234.425000,232.975000,234.487500,0.453960,0.136882,0.101749,0.118393,0.357117
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9475,0.0,2.000000,0.0,0.0,0.73,0.0,5.9400,5.6200,10.600000,236.500000,234.800000,236.700000,5.233240,1.058030,0.932165,1.845840,3.838310
9476,0.0,1.952381,0.0,0.0,0.73,0.0,5.9400,5.6125,10.600000,236.500000,234.725000,236.625000,5.233668,1.059732,0.927292,1.843703,3.831455
9477,0.0,2.000000,0.0,0.0,0.73,0.0,5.9400,5.6200,10.610000,236.400000,234.900000,236.700000,5.231420,1.063250,0.933172,1.842850,3.836670
9478,0.0,2.000000,0.0,0.0,0.78,0.0,10.0100,5.6200,10.490000,234.200000,233.200000,234.500000,6.095990,1.993200,0.925165,1.815090,4.734360


In [8]:
#drop unwanted columns
drop_cols = ['result', 'controller', 'units','table','site']
df_corr = df1.drop(columns=drop_cols, errors='ignore')
df=df_corr.copy()

Feature Engineering

In [9]:
df=df_corr.copy()
df = df.reset_index()

df["hour"] = df["time"].dt.hour
df["dayofweek"] = df["time"].dt.dayofweek
df["month"] = df["time"].dt.month
df["is_weekend"] = df["dayofweek"].isin([5,6]).astype(int)


Lag Features

In [10]:
#lag features (15-min interval)  # 1 step, 1 hour, 1 day lag
for lag in [1,4,96]:
    df[f'lag_{lag}']=df['power_total'].shift(lag)

Rolling Statistics

In [11]:
df["rolling_mean_4"] = df["power_total"].shift(1).rolling(4).mean()
df["rolling_std_4"] = df["power_total"].shift(1).rolling(4).std()

In [12]:
#Drop Nans after lagging
df=df.dropna()
df=df.drop(columns=['time'])

In [13]:
df=df.drop(columns=['index'])
df.columns

Index(['power_factor_avg', 'current_l1', 'current_l2', 'current_l3',
       'volts_l1_n', 'volts_l2_n', 'volts_l3_n', 'apparent_power_total',
       'power_l1', 'power_l2', 'power_l3', 'power_total', 'hour', 'dayofweek',
       'month', 'is_weekend', 'lag_1', 'lag_4', 'lag_96', 'rolling_mean_4',
       'rolling_std_4'],
      dtype='object')

In [14]:
target_col="power_total"
feature_cols=df.drop(columns=["power_total"]).columns

Rolling window

In [15]:
# 4. Rolling window parameters
rows_per_day = 96      # 15-minute intervals
train_days = 60
test_days = 7
train_size = train_days * rows_per_day
test_size = test_days * rows_per_day

r2_scores = []

In [24]:
# Rolling window parameters
rows_per_day = 96
train_days = 60
test_days = 7

train_size = train_days * rows_per_day
test_size = test_days * rows_per_day

# Metric storage
mae_scores = []
rmse_scores = []
mape_scores = []
r2_scores = []

for start in range(0, len(df) - train_size - test_size + 1, test_size):

    train_idx = slice(start, start + train_size)
    test_idx = slice(start + train_size, start + train_size + test_size)

    X_train = df.iloc[train_idx][feature_cols]
    y_train = df.iloc[train_idx][target_col]

    X_test = df.iloc[test_idx][feature_cols]
    y_test = df.iloc[test_idx][target_col]

    model = xgb.XGBRegressor(
        n_estimators=500,
        max_depth=5,
        learning_rate=0.05,
        objective='reg:squarederror',
        random_state=42
    )

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    # ---- METRICS ----
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    # Safe MAPE (avoid divide by zero)
    mape = np.mean(np.abs((y_test - y_pred) / (y_test + 1e-8))) * 100

    # Store
    mae_scores.append(mae)
    rmse_scores.append(rmse)
    r2_scores.append(r2)
    mape_scores.append(mape)
print("\nRolling Window Backtesting Results")
print("-"*50)
print(f"Number of splits: {len(r2_scores)}")

print("\nAverage Metrics Across All Splits")
print(f"Mean MAE  : {np.mean(mae_scores):.4f}")
print(f"Mean RMSE : {np.mean(rmse_scores):.4f}")
print(f"Mean MAPE : {np.mean(mape_scores):.2f}%")
print(f"Mean R²   : {np.mean(r2_scores):.4f}")

print("\nStandard Deviation of Metrics")
print(f"Std MAE   : {np.std(mae_scores):.4f}")
print(f"Std RMSE  : {np.std(rmse_scores):.4f}")
print(f"Std MAPE  : {np.std(mape_scores):.2f}%")
print(f"Std R²    : {np.std(r2_scores):.4f}")
print("\nSplit-wise Results")
print("-"*50)

for i in range(len(r2_scores)):
    print(f"Split {i+1}: "
          f"MAE={mae_scores[i]:.4f}, "
          f"RMSE={rmse_scores[i]:.4f}, "
          f"MAPE={mape_scores[i]:.2f}%, "
          f"R²={r2_scores[i]:.4f}")



Rolling Window Backtesting Results
--------------------------------------------------
Number of splits: 5

Average Metrics Across All Splits
Mean MAE  : 0.1028
Mean RMSE : 0.1456
Mean MAPE : 2.92%
Mean R²   : 0.8231

Standard Deviation of Metrics
Std MAE   : 0.0799
Std RMSE  : 0.1007
Std MAPE  : 2.18%
Std R²    : 0.2144

Split-wise Results
--------------------------------------------------
Split 1: MAE=0.2137, RMSE=0.2710, MAPE=6.06%, R²=0.4191
Split 2: MAE=0.1830, RMSE=0.2607, MAPE=4.89%, R²=0.7933
Split 3: MAE=0.0634, RMSE=0.1046, MAPE=2.19%, R²=0.9231
Split 4: MAE=0.0202, RMSE=0.0375, MAPE=0.64%, R²=0.9885
Split 5: MAE=0.0334, RMSE=0.0541, MAPE=0.84%, R²=0.9915


In [ ]:
print(df.dtypes)

power_factor_avg        float64
current_l1              float64
current_l2              float64
current_l3              float64
volts_l1_n              float64
volts_l2_n              float64
volts_l3_n              float64
apparent_power_total    float64
power_l1                float64
power_l2                float64
power_l3                float64
power_total             float64
hour                      int32
dayofweek                 int32
month                     int32
is_weekend                int64
lag_1                   float64
lag_4                   float64
lag_96                  float64
rolling_mean_4          float64
rolling_std_4           float64
dtype: object
